# 08 Judge calibration
Check an LLM judge against 40 human labels before you trust its scores.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-08"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: load the human labels

In [ ]:
import re
import pandas as pd
from kit.llm import call_llm
from kit.prices import cost

labels = pd.read_csv(ROOT / "evals" / "labels.csv")
print(labels.groupby(["split", "human_label"]).size().unstack())
labels.head(3)

Step 3: the judge prompt

In [ ]:
JUDGE_SYSTEM = ("You grade answers from a help assistant. PASS only if the answer is correct and fully "
                "supported by the reference. Any wrong detail, invented feature or refusal is a FAIL. "
                "Reply with PASS or FAIL, a colon, then one short reason.")

def judge(row):
    r = call_llm(f"Question: {row.question}\nReference: {row.reference}\nAnswer: {row.answer}\n"
                 "Reply with PASS or FAIL.", system=JUDGE_SYSTEM, max_output_tokens=800)
    verdict = re.match(r"\s*(PASS|FAIL)", r["text"].upper())
    return (verdict.group(1).lower() if verdict else "unparsed"), r["text"], cost(r["input_tokens"], r["output_tokens"], r["model"])

print(judge(labels.iloc[0]))

Step 4: run the judge on every label

In [ ]:
out = [judge(row) for row in labels.itertuples()]
labels["judge_label"] = [o[0] for o in out]
labels["judge_reason"] = [o[1] for o in out]
print("Judge cost for 40 labels: $", round(sum(o[2] for o in out), 5))
print("Unparsed verdicts:", int((labels.judge_label == "unparsed").sum()))

Step 5: confusion matrix and rates, dev and held-out

In [ ]:
def rates(df):
    tp = ((df.human_label == "pass") & (df.judge_label == "pass")).sum()
    tn = ((df.human_label == "fail") & (df.judge_label == "fail")).sum()
    return {"true_positive_rate": tp / (df.human_label == "pass").sum(),
            "true_negative_rate": tn / (df.human_label == "fail").sum(),
            "agreement": (df.human_label == df.judge_label).mean()}

for split in ["dev", "held-out"]:
    part = labels[labels.split == split]
    print(f"\n{split}:")
    print(pd.crosstab(part.human_label, part.judge_label, rownames=["human"], colnames=["judge"]))
    print({k: round(v, 2) for k, v in rates(part).items()})

Step 6: read the held-out disagreements

In [ ]:
held = labels[labels.split == "held-out"]
held[held.human_label != held.judge_label][["id", "answer", "human_label", "judge_label", "note"]]